In [ ]:
%%writefile bbox_mappable_matsim_to_rdf.py
#!/usr/bin/env python3
"""
matsim_to_rdf.py -- one linked TwIS RDF graph from MATSim network + plans + events.

    python3 matsim_to_rdf.py [--network network.xml] [--plans plans.xml]
                             [--events events.xml[.gz]] [--out graph.ttl]
                             [--limit N]        # cap network links (debugging)

Writes a single Turtle A-Box. All layers share IRIs so the graph is linked:
    network  ->  mun:link_<id>, mun:nodes_<id>
    plans    ->  mun:person_<id>, mun:plan_<id>, mun:activity_.., mun:leg_.., mun:route_..
    events   ->  mun:trip_<person>_<n>
Routes and executed trips reference the SAME mun:link_ ids as the network, and
plans + events share mun:person_/mun:vehicle_ ids -> cross-layer joins for free.

Every geometry is written twice: GK4 (EPSG:31468, authoritative) + CRS84
(WGS84 lon/lat, for GraphDB's index/map). Load the T-Box
(twis.ttl + twis_demand.ttl + twis_events.ttl) alongside for reasoning.
"""
import sys, gzip, json, time, argparse
from lxml import etree
from pyproj import Transformer

CRS   = "http://www.opengis.net/def/crs/EPSG/0/31468"
CRS84 = "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
_tw = Transformer.from_crs("EPSG:31468", "EPSG:4326", always_xy=True)   # -> lon,lat
def wgs(x, y):
    lon, lat = _tw.transform(float(x), float(y)); return f"{lon:.7f} {lat:.7f}"
def dbl(v): return f'"{v}"^^xsd:double'
def opn(p): return gzip.open(p, "rb") if p.endswith(".gz") else open(p, "rb")

PREFIXES = """@prefix twis: <https://w3id.org/twis#> .
@prefix mun:  <https://example.org/matsim/munich/> .
@prefix geo:  <http://www.opengis.net/ont/geosparql#> .
@prefix sf:   <http://www.opengis.net/ont/sf#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

"""

# ---------------------------------------------------------------- NETWORK ----
def convert_network(src, out, limit=None, bbox=None):
    coords, emitted = {}, set()
    n_nodes = n_links = 0
    def emit_node(nid):
        nonlocal n_nodes
        if nid in emitted: return
        x, y = coords[nid]
        out.write(f"mun:nodes_{nid} a twis:Nodes, geo:Feature ;\n"
                  f"    geo:hasGeometry mun:geom_nodes_{nid}, mun:geowgs_nodes_{nid} .\n"
                  f'mun:geom_nodes_{nid} a sf:Point ; geo:asWKT "<{CRS}> POINT({x} {y})"^^geo:wktLiteral .\n'
                  f'mun:geowgs_nodes_{nid} a sf:Point ; geo:asWKT "<{CRS84}> POINT({wgs(x,y)})"^^geo:wktLiteral .\n')
        emitted.add(nid); n_nodes += 1
    ctx = etree.iterparse(opn(src), events=("end",), tag=("node", "link"))
    for _, e in ctx:
        if e.tag == "node":
            keep = True
            if bbox is not None:
                lon, lat = _tw.transform(float(e.get("x")), float(e.get("y")))
                keep = (bbox[0] <= lon <= bbox[1] and bbox[2] <= lat <= bbox[3])
            if keep:
                coords[e.get("id")] = (e.get("x"), e.get("y"))
        else:
            if limit is not None and n_links >= limit:
                e.clear(); break
            lid, frm, to = e.get("id"), e.get("from"), e.get("to")
            if frm not in coords or to not in coords:   # link outside the box -> skip
                e.clear()
                while e.getprevious() is not None: del e.getparent()[0]
                continue
            at = {a.get("name"): a.text for a in e.iter("attribute")}
            fx, fy = coords.get(frm, ("", "")); tx, ty = coords.get(to, ("", ""))
            w84 = f"<{CRS84}> LINESTRING({wgs(fx,fy)}, {wgs(tx,ty)})" if fx and tx else None
            out.write(f"mun:link_{lid} a twis:Ast, geo:Feature ;\n"
                      f"    twis:length {dbl(e.get('length'))} ; twis:freespeed {dbl(e.get('freespeed'))} ;\n"
                      f"    twis:capacity {dbl(e.get('capacity'))} ; twis:permlanes {dbl(e.get('permlanes'))} ;\n"
                      f'    twis:oneway "{e.get("oneway")}" ; twis:modes "{e.get("modes")}" ;\n'
                      f'    twis:classification "{at.get("type","")}" ; twis:origid "{at.get("origid","")}" ;\n'
                      f"    twis:fromNodes mun:nodes_{frm} ; twis:toNodes mun:nodes_{to} ;\n"
                      f"    geo:hasGeometry mun:geom_link_{lid}" + (f", mun:geowgs_ast_{lid} .\n" if w84 else " .\n") +
                      f'mun:geom_link_{lid} a sf:LineString ; geo:asWKT "<{CRS}> LINESTRING({fx} {fy}, {tx} {ty})"^^geo:wktLiteral .\n' +
                      (f'mun:geowgs_ast_{lid} a sf:LineString ; geo:asWKT "{w84}"^^geo:wktLiteral .\n' if w84 else "") +
                      f'mun:linkport_{lid}_A a twis:Linkport ; twis:status "Start" ; twis:ast mun:link_{lid} ; twis:nodes mun:nodes_{frm} .\n'
                      f'mun:linkport_{lid}_E a twis:Linkport ; twis:status "End" ; twis:ast mun:link_{lid} ; twis:nodes mun:nodes_{to} .\n')
            emit_node(frm); emit_node(to); n_links += 1
        e.clear()
        while e.getprevious() is not None: del e.getparent()[0]
    return f"network: {n_nodes} Nodes, {n_links} Links"

# ------------------------------------------------------------------ PLANS ----
def _child_attr(elem, name):
    for a in elem.findall("./attributes/attribute"):
        if a.get("name") == name: return a.text
    return None

def convert_plans(src, out):
    np = npl = na = nl = 0
    ctx = etree.iterparse(opn(src), events=("end",), tag="person")
    for _, person in ctx:
        pid = person.get("id")
        veh = []
        raw = _child_attr(person, "vehicles")
        if raw:
            try: veh = list(json.loads(raw).values())
            except Exception: pass
        out.write(f"mun:person_{pid} a twis:Person .\n")
        for v in veh:
            out.write(f"mun:vehicle_{v} a twis:Vehicle .\nmun:person_{pid} twis:automobile mun:vehicle_{v} .\n")
        np += 1
        for pi, plan in enumerate(person.findall("./plan"), 1):
            plan_id = f"{pid}_{pi}"
            sel = str(plan.get("selected") == "yes").lower()
            out.write(f"mun:plan_{plan_id} a twis:Version ; twis:belongsToPerson mun:person_{pid} ;\n"
                      f'    twis:selected "{sel}"^^xsd:boolean')
            out.write(f" ; twis:score {dbl(plan.get('score'))} .\n" if plan.get("score") else " .\n")
            npl += 1
            last_act, pending, seq = None, None, 0
            for el in plan:
                tag = etree.QName(el).localname
                if tag == "activity":
                    aid = f"{plan_id}_{seq}"; typ = el.get("type"); lk = el.get("link")
                    x, y = el.get("x"), el.get("y")
                    out.write(f'mun:activity_{aid} a twis:Activity ; twis:typ "{typ}" ;\n'
                              f"    twis:inVersion mun:plan_{plan_id} ; twis:position {seq}")
                    if lk: out.write(f" ;\n    twis:liesOnLink mun:link_{lk}")
                    if el.get("end_time"): out.write(f' ;\n    twis:endTime "{el.get("end_time")}"')
                    if x and y:
                        out.write(f" ;\n    geo:hasGeometry mun:geom_act_{aid}, mun:geowgs_act_{aid} .\n"
                                  f'mun:geom_act_{aid} a sf:Point ; geo:asWKT "<{CRS}> POINT({x} {y})"^^geo:wktLiteral .\n'
                                  f'mun:geowgs_act_{aid} a sf:Point ; geo:asWKT "<{CRS84}> POINT({wgs(x,y)})"^^geo:wktLiteral .\n')
                    else:
                        out.write(" .\n")
                    if pending is not None:
                        out.write(f"mun:leg_{pending} twis:toActivity mun:activity_{aid} .\n"); pending = None
                    last_act = aid; na += 1; seq += 1
                elif tag == "leg":
                    lid = f"{plan_id}_{seq}"; rid = lid
                    out.write(f'mun:leg_{lid} a twis:Leg ; twis:modus "{el.get("mode")}" ;\n'
                              f"    twis:inVersion mun:plan_{plan_id} ; twis:position {seq}")
                    if el.get("dep_time"): out.write(f' ;\n    twis:departure "{el.get("dep_time")}"')
                    if el.get("trav_time"): out.write(f' ;\n    twis:travelTime "{el.get("trav_time")}"')
                    if last_act: out.write(f" ;\n    twis:fromActivity mun:activity_{last_act}")
                    out.write(f" ;\n    twis:route mun:route_{rid} .\n")
                    r = el.find("./route")
                    if r is not None:
                        links = r.text.split() if r.text else []
                        out.write(f'mun:route_{rid} a twis:Route ; twis:typ "{r.get("type")}"')
                        if r.get("start_link"): out.write(f" ;\n    twis:startLink mun:link_{r.get('start_link')}")
                        if r.get("end_link"):   out.write(f" ;\n    twis:endLink mun:link_{r.get('end_link')}")
                        if r.get("distance"):   out.write(f" ;\n    twis:distance {dbl(r.get('distance'))}")
                        if r.get("vehicleRefId"): out.write(f" ;\n    twis:automobile mun:vehicle_{r.get('vehicleRefId')}")
                        out.write(f' ;\n    twis:linkJoins "{" ".join(links)}"')
                        if links: out.write(" ;\n    twis:usesLink " + ", ".join(f"mun:link_{l}" for l in links))
                        out.write(" .\n")
                    pending = lid; nl += 1; seq += 1
        person.clear()
        while person.getprevious() is not None: del person.getparent()[0]
    return f"plans: {np} Person, {npl} Version, {na} Activity, {nl} Leg"

# ----------------------------------------------------------------- EVENTS ----
def convert_events(src, out):
    veh2p, legs, seq, ntr = {}, {}, {}, 0
    def finish(p, arr, end_link):
        nonlocal ntr
        leg = legs.pop(p, None)
        if not leg: return
        seq[p] = seq.get(p, 0) + 1
        tid = f"{p}_{seq[p]}"
        trav = leg["trav"]; links = [r[0] for r in trav]
        times = [str(int(r[2]-r[1])) if r[2] is not None else "" for r in trav]
        dur = arr - leg["dep"]
        out.write(f"mun:person_{p} a twis:Person .\nmun:vehicle_{p} a twis:Vehicle .\n"
                  f"mun:trip_{tid} a twis:TripCompleted ;\n"
                  f"    twis:person mun:person_{p} ; twis:automobile mun:vehicle_{p} ;\n"
                  f'    twis:modus "{leg["mode"]}" ;\n'
                  f'    twis:startSec "{leg["dep"]}"^^xsd:double ; twis:endSec "{arr}"^^xsd:double ;\n'
                  f'    twis:durationSec "{dur}"^^xsd:double ; twis:noOfLinks {len(links)}')
        if links:
            out.write(f" ;\n    twis:startLink mun:link_{links[0]} ; twis:endLink mun:link_{end_link} ;\n"
                      f'    twis:linkJoins "{" ".join(links)}" ; twis:travelTimeJoins "{" ".join(times)}" ;\n'
                      f"    twis:usesLink " + ", ".join(f"mun:link_{l}" for l in links) + " .\n")
        else:
            out.write(" .\n")
        ntr += 1
    ctx = etree.iterparse(opn(src), events=("end",), tag="event")
    for _, e in ctx:
        typ = e.get("type"); t = float(e.get("time"))
        if typ == "departure":
            legs[e.get("person")] = {"mode": e.get("legMode"), "dep": t, "trav": []}
        elif typ == "PersonEntersVehicle":
            veh2p[e.get("vehicle")] = e.get("person")
        elif typ in ("vehicle enters traffic", "entered link"):
            p = veh2p.get(e.get("vehicle"))
            if p in legs: legs[p]["trav"].append([e.get("link"), t, None])
        elif typ in ("left link", "vehicle leaves traffic"):
            p = veh2p.get(e.get("vehicle"))
            if p in legs and legs[p]["trav"]: legs[p]["trav"][-1][2] = t
        elif typ == "PersonLeavesVehicle":
            veh2p.pop(e.get("vehicle"), None)
        elif typ == "arrival":
            finish(e.get("person"), t, e.get("link"))
        e.clear()
        while e.getprevious() is not None: del e.getparent()[0]
    return f"events: {ntr} TripCompleted"

# -------------------------------------------------------------------- MAIN ----
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--network"); ap.add_argument("--plans"); ap.add_argument("--events")
    ap.add_argument("--out", default="..../MATSim_graph1000.ttl")
    ap.add_argument("--limit", type=int)
    ap.add_argument("--bbox", help="LON_MIN,LON_MAX,LAT_MIN,LAT_MAX (WGS84)")
    a = ap.parse_args()
    bbox = tuple(float(v) for v in a.bbox.split(',')) if a.bbox else None
    if not any([a.network, a.plans, a.events]):
        ap.error("give at least one of --network / --plans / --events")
    t0 = time.time()
    with open(a.out, "w") as out:
        out.write(PREFIXES)
        if a.network: print(convert_network(a.network, out, a.limit, bbox))
        if a.plans:   print(convert_plans(a.plans, out))
        if a.events:  print(convert_events(a.events, out))
    print(f"-> {a.out}   ({time.time()-t0:.1f}s)")

Writing bbox_mappable_matsim_to_rdf.py


In [ ]:
!pip install lxml pyproj rdflib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 27.9 MB/s eta 0:00:00


In [ ]:
!python bbox_mappable_matsim_to_rdf.py \
  --network "..../munich.output_network.xml.gz" \
  --bbox "11.53,11.60,48.12,48.16" \
  #--plans   ".../munich.output_plans.xml.gz" \
  #--events  "..../munich.output_events.xml.gz" \
  --out     "/content/MATSim_graph1000_network.ttl"

network: 2513 Nodes, 5203 Links
-> /content/drive/MyDrive/Thesis_TUM/MATSim_graph1000.ttl   (5.2s)
